# Selective next-FX-renko-bar direction research

This notebook consumes the reusable `range_bars_ml` library; FX feature construction and training logic do not live here.


In [1]:
# Reload local modelling modules so re-running this cell picks up source edits.
import importlib
import range_bars_ml as _range_bars_package
import range_bars_ml.model as _range_bars_model
import range_bars_ml.pipeline as _range_bars_pipeline
import range_bars_ml.fx_features as _range_bars_fx_features
import range_bars_ml.fx_pipeline as _range_bars_fx_pipeline

importlib.reload(_range_bars_model)
importlib.reload(_range_bars_pipeline)
importlib.reload(_range_bars_fx_features)
importlib.reload(_range_bars_fx_pipeline)
importlib.reload(_range_bars_package)

from range_bars_ml import (
    ExperimentConfig,
    FxFeatureConfig,
    ModelConfig,
    load_fx_renko_bars,
    prepare_fx_dataset,
    run_fx_experiment,
)
from range_bars_ml.model import signal_metrics
import numpy as np
import polars as pl


In [2]:
# Data selection
DATA_PATH = r"C:/Code/Trading/2026/ml/data/eurusd_renko_10bps.parquet"  # Set this to your FX parquet file.

# Causal feature definition
FEATURE_CONFIG = FxFeatureConfig(
    windows=range(1,30,1),
    time_column="close_timestamp",
)

# LightGBM and probability-calibration definition
MODEL_CONFIG = ModelConfig(
    calibration_fraction=0.10,
    random_state=18616,
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=100,
)

# Require a probability edge above the development-period up rate.
# Set to 0.0 to disable this additional anti-trend gate.
EDGE_MARGIN = 0.01

# Chronological evaluation and selective-signal definition
EXPERIMENT_CONFIG = ExperimentConfig(
    feature=FEATURE_CONFIG,
    model=MODEL_CONFIG,
    test_fraction=0.20,
    n_folds=10,
    # Zero permits an all-abstain strategy if no calls have positive net score.
    minimum_coverage=0.0,
    abstention_penalty=-0.0001,
    edge_margin=EDGE_MARGIN,
)

# Inspection and artifact choices
FEATURE_PREVIEW_ROWS = 10
FEATURE_EXPORT_PATH = None  # e.g. r"C:\Code\Trading\2026\ml\data\fx_renko_features.parquet"
FEATURE_EXPORT_COMPRESSION = "zstd"
MODEL_ARTIFACT_PATH = "fx_renko_model.joblib"


In [3]:
all_bars = load_fx_renko_bars(DATA_PATH)
all_bars = all_bars.rename({'close_time':'close_timestamp'})


In [4]:
bars = all_bars.filter(pl.col("close_timestamp") < pl.datetime(2025, 1, 1, time_zone='UTC'))
#bars = bars.slice(offset=DATA_SLICE_OFFSET, length=DATA_SLICE_LENGTH)

In [5]:
bars.shape

(145988, 6)

In [6]:
bars[-1]

timestamp,close_timestamp,mid_open,mid_high,mid_low,mid_close
"datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,f64,f64
2024-12-31 19:16:47.847 UTC,2024-12-31 21:58:30.421 UTC,1.036425,1.03647,1.03537,1.03537


In [7]:
# Inspect the fully causal, next-bar-labelled training frame.
feature_df, feature_names = prepare_fx_dataset(bars, config=FEATURE_CONFIG)
feature_df.head(FEATURE_PREVIEW_ROWS)

# Set FEATURE_EXPORT_PATH above to persist the complete feature frame.
if FEATURE_EXPORT_PATH:
    feature_df.write_parquet(FEATURE_EXPORT_PATH, compression=FEATURE_EXPORT_COMPRESSION)


In [8]:
feature_df

bar_return,bar_range_pct,close_in_range,upper_wick_share,lower_wick_share,body_to_range,bar_direction,log_bar_duration,close_hour_sin,close_hour_cos,close_weekday_sin,close_weekday_cos,log_return_1,return_mean_1,return_std_1,direction_persistence_1,reversal_rate_1,direction_streak_share_1,trend_strength_1,return_acceleration_1,close_mean_zscore_1,channel_position_1,range_mean_1,range_std_1,range_regime_1,duration_log_mean_1,duration_log_std_1,duration_log_zscore_1,bar_time_share_window_1_lag_0,bar_is_up_window_1_lag_0,log_return_2,return_mean_2,return_std_2,direction_persistence_2,reversal_rate_2,direction_streak_share_2,trend_strength_2,…,bar_time_share_window_29_lag_11,bar_is_up_window_29_lag_11,bar_time_share_window_29_lag_12,bar_is_up_window_29_lag_12,bar_time_share_window_29_lag_13,bar_is_up_window_29_lag_13,bar_time_share_window_29_lag_14,bar_is_up_window_29_lag_14,bar_time_share_window_29_lag_15,bar_is_up_window_29_lag_15,bar_time_share_window_29_lag_16,bar_is_up_window_29_lag_16,bar_time_share_window_29_lag_17,bar_is_up_window_29_lag_17,bar_time_share_window_29_lag_18,bar_is_up_window_29_lag_18,bar_time_share_window_29_lag_19,bar_is_up_window_29_lag_19,bar_time_share_window_29_lag_20,bar_is_up_window_29_lag_20,bar_time_share_window_29_lag_21,bar_is_up_window_29_lag_21,bar_time_share_window_29_lag_22,bar_is_up_window_29_lag_22,bar_time_share_window_29_lag_23,bar_is_up_window_29_lag_23,bar_time_share_window_29_lag_24,bar_is_up_window_29_lag_24,bar_time_share_window_29_lag_25,bar_is_up_window_29_lag_25,bar_time_share_window_29_lag_26,bar_is_up_window_29_lag_26,bar_time_share_window_29_lag_27,bar_is_up_window_29_lag_27,bar_time_share_window_29_lag_28,bar_is_up_window_29_lag_28,target
f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,f64,f64,…,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,f64,i8,i8
-0.001001,0.001838,0.0,0.455422,0.0,0.544578,-1,11.313779,-0.965926,-0.258819,0.781831,0.62349,-0.0012,-0.0012,0.0,-1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.001838,0.0,0.0,11.313779,0.0,0.0,1.0,0,-0.000598,-0.000299,0.000901,0.0,0.5,0.5,-0.469276,…,0.008055,0,0.001328,0,0.000766,1,0.021727,1,0.002219,1,0.011286,1,0.039381,0,0.036025,1,0.005161,1,0.010859,0,0.020074,1,0.109798,0,0.197905,1,0.01905,0,0.022895,1,0.028937,0,0.164877,1,0.193847,0,1
0.001081,0.001312,1.0,0.0,0.175676,0.824324,1,14.239981,-1.0,-1.8370e-16,0.781831,0.62349,0.001138,0.001138,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.001312,0.0,0.0,14.239981,0.0,0.0,1.0,1,-0.000062,-0.000031,0.001169,0.0,1.0,0.5,-0.037476,…,0.001013,0,0.009715,0,0.001601,0,0.000924,1,0.026207,1,0.002677,1,0.013613,1,0.0475,0,0.043452,1,0.006225,1,0.013098,0,0.024213,1,0.132435,0,0.238708,1,0.022977,0,0.027615,1,0.034902,0,0.198871,1,0
-0.001058,0.0012,0.0,0.118081,0.0,0.881919,-1,14.451807,-1.0,-1.8370e-16,0.781831,0.62349,-0.00124,-0.00124,0.0,-1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0012,0.0,0.0,14.451807,0.0,0.0,1.0,0,-0.000102,-0.000051,0.001189,0.0,1.0,0.5,-0.060608,…,0.001081,0,0.001212,0,0.011631,0,0.001917,0,0.001106,1,0.031374,1,0.003205,1,0.016297,1,0.056866,0,0.05202,1,0.007453,1,0.01568,0,0.028987,1,0.15855,0,0.285778,1,0.027508,0,0.03306,1,0.041785,0,0
-0.001041,0.001041,0.0,0.0,0.0,1.0,-1,6.942157,-1.0,-1.8370e-16,0.781831,0.62349,-0.000168,-0.000168,0.0,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.001041,0.0,0.0,6.942157,0.0,0.0,1.0,0,-0.001409,-0.000704,0.000536,-1.0,0.5,1.0,-1.858709,…,0.002067,1,0.001128,0,0.001265,0,0.012138,0,0.002,0,0.001154,1,0.032742,1,0.003344,1,0.017007,1,0.059345,0,0.054287,1,0.007778,1,0.016364,0,0.030251,1,0.16546,0,0.298233,1,0.028707,0,0.034501,1,1
0.001006,0.001006,1.0,0.0,0.0,1.0,1,9.288967,-1.0,-1.8370e-16,0.781831,0.62349,0.000979,0.000979,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.001006,0.0,0.0,9.288967,0.0,0.0,1.0,1,0.000811,0.000405,0.000574,0.0,0.5,0.5,0.999088,…,0.006238,1,0.00214,1,0.001168,0,0.00131,0,0.012568,0,0.002071,0,0

In [9]:
# Metrics include score (+1 correct, -1 wrong, -0.001 abstain).
result = run_fx_experiment(bars, config=EXPERIMENT_CONFIG)

# The edge gate is applied inside run_fx_experiment before its single final-test score.
dataset, _ = prepare_fx_dataset(bars, config=FEATURE_CONFIG)
test = dataset[result.split_plan.test.tolist()]
development_up_rate = result.model.metadata["development_up_rate"]


In [10]:
# Compare against simple regime/trend baselines on the same final test.
test_target = test["target"].to_numpy()
baseline_metrics = {
    "always_up": signal_metrics(np.ones(len(test_target)), test_target, 0.5, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_down": signal_metrics(np.zeros(len(test_target)), test_target, 1.01, 0.5, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_abstain": signal_metrics(np.full(len(test_target), 0.5), test_target, 1.01, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
}
score_comparison = pl.DataFrame([
    {"strategy": "model_edge_gated", **result.test_metrics},
    *({"strategy": name, **metrics} for name, metrics in baseline_metrics.items()),
]).select(["strategy", "score", "mean_score", "signals", "correct", "wrong", "abstentions", "precision"])

print(f"Development up rate: {development_up_rate:.3%}; edge gates: long >= {result.model.long_threshold:.3f}, short <= {result.model.short_threshold:.3f}")
score_comparison

Development up rate: 50.240%; edge gates: long >= 0.512, short <= 0.492


strategy,score,mean_score,signals,correct,wrong,abstentions,precision
str,f64,f64,f64,f64,f64,f64,f64
"""model_edge_gated""",332.0,0.011373,29192.0,14762.0,14430.0,0.0,0.505686
"""always_up""",-42.0,-0.001439,29192.0,14575.0,14617.0,0.0,0.499281
"""always_down""",42.0,0.001439,29192.0,14617.0,14575.0,0.0,0.500719
"""always_abstain""",-2.9192,-0.0001,0.0,0.0,0.0,29192.0,NaN


In [11]:
result.model.save(MODEL_ARTIFACT_PATH)


In [12]:
from range_bars_ml.model import SelectiveSignalModel
model = SelectiveSignalModel.load(MODEL_ARTIFACT_PATH)
feature_config = FxFeatureConfig(**model.metadata["feature_config"])

In [13]:
new_bars = all_bars.filter(pl.col("close_timestamp") >= pl.datetime(2025, 1, 1, time_zone='UTC'))
dataset, _ = prepare_fx_dataset(new_bars, feature_config)

In [14]:
predictions = model.predict(dataset)
signals = dataset.select("target").hstack(predictions)

In [15]:
metrics = signal_metrics(
      signals["probability_up"].to_numpy(),
      signals["target"].to_numpy(),
      model.long_threshold,
      model.short_threshold,
      abstention_penalty=model.metadata.get("abstention_penalty", -0.001),
  )

In [16]:
print(metrics)

{'coverage': 0.6215472916417554, 'signals': 5198.0, 'correct': 2613.0, 'wrong': 2585.0, 'abstentions': 3165.0, 'score': 27.6835, 'mean_score': 0.003310235561401411, 'precision': 0.5026933435936899, 'long_coverage': 0.11873729522898481, 'long_precision': 0.5186304128902316, 'long_signals': 993.0, 'short_coverage': 0.5028099964127706, 'short_precision': 0.49892984542211655, 'short_signals': 4205.0}


In [17]:
feature_df.columns

['bar_return',
 'bar_range_pct',
 'close_in_range',
 'upper_wick_share',
 'lower_wick_share',
 'body_to_range',
 'bar_direction',
 'log_bar_duration',
 'close_hour_sin',
 'close_hour_cos',
 'close_weekday_sin',
 'close_weekday_cos',
 'log_return_1',
 'return_mean_1',
 'return_std_1',
 'direction_persistence_1',
 'reversal_rate_1',
 'direction_streak_share_1',
 'trend_strength_1',
 'return_acceleration_1',
 'close_mean_zscore_1',
 'channel_position_1',
 'range_mean_1',
 'range_std_1',
 'range_regime_1',
 'duration_log_mean_1',
 'duration_log_std_1',
 'duration_log_zscore_1',
 'bar_time_share_window_1_lag_0',
 'bar_is_up_window_1_lag_0',
 'log_return_2',
 'return_mean_2',
 'return_std_2',
 'direction_persistence_2',
 'reversal_rate_2',
 'direction_streak_share_2',
 'trend_strength_2',
 'return_acceleration_2',
 'close_mean_zscore_2',
 'channel_position_2',
 'range_mean_2',
 'range_std_2',
 'range_regime_2',
 'duration_log_mean_2',
 'duration_log_std_2',
 'duration_log_zscore_2',
 'bar_ti

In [20]:
bars[-1]


timestamp,close_timestamp,mid_open,mid_high,mid_low,mid_close
"datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,f64,f64
2024-12-31 19:16:47.847 UTC,2024-12-31 21:58:30.421 UTC,1.036425,1.03647,1.03537,1.03537


In [21]:
import pandas as pd
from range_bars_ml import SelectiveSignalModel

model = SelectiveSignalModel.load(MODEL_ARTIFACT_PATH)

importance = (
  pd.DataFrame({
      "feature": model.feature_columns,
      "split_count": model.estimator.feature_importances_,
      "gain": model.estimator.booster_.feature_importance(importance_type="gain"),
  })
  .sort_values("gain", ascending=False)
)

print(importance.head(30).to_string(index=False))

                        feature  split_count        gain
                   log_return_1          200 7376.335473
                     bar_return          310 4866.229233
                   log_return_2          230 3365.439279
                 close_in_range          143 2653.031558
                 close_hour_cos          163 2304.532218
                  bar_range_pct          167 2104.368207
                   return_std_2          159 2019.184398
bar_time_share_window_29_lag_28          175 1986.680152
                   return_std_4          152 1921.629018
                   return_std_3          151 1728.244863
                   log_return_3          139 1648.395160
                   log_return_5          106 1321.049449
                 close_hour_sin          104 1308.725579
                   log_return_4          110 1295.476262
bar_time_share_window_28_lag_27          116 1274.932765
                   return_std_6          104 1262.829137
                   return_std_5